In [3]:
import pandas as pd
import random
from faker import Faker

fake = Faker("en_NG")
num_records = 100

data = []

# Valid Nigerian prefixes
valid_prefixes = ["070", "080", "081", "090", "091", "071"]

def generate_valid_phone():
    prefix = random.choice(valid_prefixes)
    remaining_digits = "".join(str(random.randint(0, 9)) for _ in range(8))
    return prefix + remaining_digits  # total = 11 digits

for _ in range(num_records):
    record = {
        "name": fake.name() if random.random() > 0.1 else None,  # 10% missing

        # ✅ Phone numbers: valid or missing only
        "phone_number": (
            generate_valid_phone() if random.random() > 0.1 else None
        ),

        # IDs: mostly valid, some invalid
        "id_number": (
            str(fake.random_number(digits=11, fix_len=True))
            if random.random() > 0.2
            else str(fake.random_number(digits=random.choice([8, 9, 12, 13])))
        ),

        "address": fake.address() if random.random() > 0.1 else None,

        "reg_date": (
            fake.date_between(start_date="-2y", end_date="today")
            if random.random() > 0.1 else None
        )
    }

    data.append(record)

df_raw = pd.DataFrame(data)

# Save raw data
df_raw.to_csv("sim_registration_raw.csv", index=False)

print("Raw fake data created successfully")
print(df_raw.head(10))


Raw fake data created successfully
                name phone_number      id_number  \
0     Joseph Olawale  07146418000    55269876055   
1      Sarah Ekwueme  09197828782    15539481444   
2         Juliet Obi  09017611033    36617908495   
3      Joshua Oyekan  07003575253    72364077869   
4    Juliet Ogunleye  07175976577    57936162634   
5   Patience Balogun  09167892219    38044504723   
6           Paul Obi  08188921878    27273866745   
7         Peter Uche  07130065912    42761626414   
8  Philip Adetokunbo         None  8311820679622   
9     Gloria Balogun  09172068959    24604493460   

                                             address    reg_date  
0        6013 Joshua Crossing\nIbrahimstad, TN 86103  2025-03-11  
1               4551 Eze Highway\nObishire, IA 42191  2025-03-11  
2            45895 Olawale Avenue\nHopeton, WI 68619  2025-07-27  
3         209 Peace Row Apt. 235\nJameston, MO 12061  2025-02-14  
4             8367 Obi Drive\nAkinwaleside, MI 42682  202

In [4]:
print(df_raw.columns)
print(df_raw["phone_number"].head(10))


Index(['name', 'phone_number', 'id_number', 'address', 'reg_date'], dtype='object')
0    07146418000
1    09197828782
2    09017611033
3    07003575253
4    07175976577
5    09167892219
6    08188921878
7    07130065912
8           None
9    09172068959
Name: phone_number, dtype: object


In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df_raw, title="Profiling Report")
profile

NameError: name 'df' is not defined

In [7]:
import pandas as pd
import sqlite3
import re

# -------------------------
# 1. EXTRACTION
# -------------------------
print("\n1️⃣ EXTRACTING DATA...")

csv_path = "sim_registration_raw.csv"

df = pd.read_csv(
    csv_path,
    dtype={"phone_number": str, "id_number": str},
    na_values=["", "NaN", "NA", "None", "nan"]
)


print(f"Rows extracted: {len(df):,}")
print(df.head())

# -------------------------
# 2. TRANSFORMATION
# -------------------------
print("\n2️⃣ TRANSFORMING DATA...")

# -------- NAME CLEANING --------
df["name"] = (
    df["name"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# -------- PHONE NUMBER CLEANING --------
def clean_phone(x):
    if pd.isna(x):
        return "Unknown"

    x = str(x).strip()

    # Remove trailing .0 if it exists
    if x.endswith(".0"):
        x = x[:-2]

    digits = re.sub(r"\D", "", x)

    # Convert Nigerian country code to local
    if digits.startswith("234") and len(digits) > 3:
        digits = "0" + digits[3:]

    if len(digits) == 11 and digits.startswith("0"):
        return digits

    return "Unknown"

df["phone_number"] = df["phone_number"].apply(clean_phone)

# -------- ID NUMBER CLEANING --------
def clean_id(x):
    if pd.isna(x):
        return "Unknown"

    digits = re.sub(r"\D", "", str(x))

    if len(digits) == 11:
        return digits

    return "Unknown"

df["id_number"] = df["id_number"].apply(clean_id)

# -------- ADDRESS CLEANING --------
df["address"] = (
    df["address"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

# -------- REGISTRATION DATE --------
df["reg_date"] = pd.to_datetime(
    df["reg_date"],
    errors="coerce"
).dt.date

print("Transformation completed")

# -------------------------
# 3. LOAD (CSV)
# -------------------------
print("\n3️⃣ LOADING CLEAN DATA TO CSV...")

clean_csv = "sim_registration_cleaned.csv"
df.to_csv(clean_csv, index=False)

print(f"Clean CSV saved: {clean_csv}")
print(f"Final rows: {len(df):,}")

# -------------------------
# 4. LOAD (DATABASE)
# -------------------------
print("\n4️⃣ LOADING DATA TO DATABASE...")

conn = sqlite3.connect("sim_registration.db")

df.to_sql(
    "sim_registrations",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Data successfully loaded into SQLite database")



1️⃣ EXTRACTING DATA...
Rows extracted: 100
              name phone_number    id_number  \
0   Joseph Olawale  07146418000  55269876055   
1    Sarah Ekwueme  09197828782  15539481444   
2       Juliet Obi  09017611033  36617908495   
3    Joshua Oyekan  07003575253  72364077869   
4  Juliet Ogunleye  07175976577  57936162634   

                                       address    reg_date  
0  6013 Joshua Crossing\nIbrahimstad, TN 86103  2025-03-11  
1         4551 Eze Highway\nObishire, IA 42191  2025-03-11  
2      45895 Olawale Avenue\nHopeton, WI 68619  2025-07-27  
3   209 Peace Row Apt. 235\nJameston, MO 12061  2025-02-14  
4       8367 Obi Drive\nAkinwaleside, MI 42682  2024-04-30  

2️⃣ TRANSFORMING DATA...
Transformation completed

3️⃣ LOADING CLEAN DATA TO CSV...
Clean CSV saved: sim_registration_cleaned.csv
Final rows: 100

4️⃣ LOADING DATA TO DATABASE...
Data successfully loaded into SQLite database
